# DEMO 5: DAX vs SQL Examples

This demo walks through 10 common DAX patterns and their SQL equivalents. If you've been writing DAX measures in Power BI, these translations will help you think in SQL.

**Key insight:** DAX operates on filter context and implicit relationships. SQL requires you to be explicit about everything — which makes the logic transparent and debuggable.

---

In [0]:
%run ./demo_5_setup

## Our Data

The setup created two tables in your personal schema:

- **`gold_monthly_sales`** — 480 rows of monthly revenue data (Jan 2023 – Dec 2024) broken down by region and product category
- **`dim_customers`** — 64 customers with region and segment attributes

Think of `gold_monthly_sales` as the equivalent of a Power BI fact table, and `dim_customers` as a dimension table you’d normally connect via a relationship.

---

## SQL Building Blocks: SELECT, FROM, WHERE

Before we start translating DAX patterns, here are the three clauses you’ll see in every query:

| Clause | What it does | Power BI analogy |
| --- | --- | --- |
| **SELECT** | Chooses which columns/calculations to return | Choosing fields in a visual |
| **FROM** | Specifies which table to read | Choosing a dataset |
| **WHERE** | Filters rows before any calculation | A slicer or filter on the visual |

`SUM()`, `AVG()`, `COUNT()` work exactly as you’d expect — they aggregate a column. The syntax is nearly identical to DAX.

Patterns 1–3 below use only these basics.

---

In [0]:
%sql
-- ============================================================
-- DAX vs SQL: Pattern 1 — Basic Aggregation
-- ============================================================
-- DAX: Total Revenue = SUM(Sales[total_revenue])
-- SQL equivalent:

SELECT SUM(total_revenue) AS total_revenue
FROM gold_monthly_sales;

In [0]:
%sql
-- ============================================================
-- DAX vs SQL: Pattern 2 — Filtered Aggregation (CALCULATE equivalent)
-- ============================================================
-- DAX: North Revenue = CALCULATE(SUM(Sales[total_revenue]), Sales[region] = "North")
--
-- In DAX, CALCULATE modifies the filter context.
-- In SQL, we simply add a WHERE clause.

SELECT SUM(total_revenue) AS north_revenue
FROM gold_monthly_sales
WHERE region = 'North';

In [0]:
%sql
-- ============================================================
-- DAX vs SQL: Pattern 3 — CALCULATE with Multiple Filters
-- ============================================================
-- DAX: 
--   North Electronics Revenue = 
--   CALCULATE(
--     SUM(Sales[total_revenue]),
--     Sales[region] = "North",
--     Sales[product_category] = "Electronics"
--   )
--
-- SQL: Multiple conditions in WHERE (combined with AND)

SELECT SUM(total_revenue) AS north_electronics_revenue
FROM gold_monthly_sales
WHERE region = 'North'
  AND product_category = 'Electronics';

## New Concept: GROUP BY and ORDER BY

In Power BI, when you drag a dimension (like Region) onto Rows and a measure (like Total Revenue) onto Values, Power BI automatically groups the data for you.

In SQL, you do this explicitly with **GROUP BY**:

- **`GROUP BY region`** tells SQL: “collapse all rows into one row per region, then apply the aggregate functions (`SUM`, `AVG`, etc.) within each group.”
- **`ORDER BY total_revenue DESC`** sorts the results. `DESC` means largest first; `ASC` (the default) means smallest first.

**Rule:** Every non-aggregated column in your SELECT must also appear in your GROUP BY. If you select `region` and `SUM(total_revenue)`, then `region` must be in the GROUP BY.

---

In [0]:
%sql
-- ============================================================
-- DAX vs SQL: Pattern 4 — Aggregation by Category
-- ============================================================
-- In Power BI, you drag a measure to Values and a dimension to Rows.
-- DAX implicitly groups by whatever is on Rows/Columns.
--
-- In SQL, you explicitly state the GROUP BY.
-- This is equivalent to a matrix visual showing Revenue by Region.

SELECT 
  region,
  SUM(total_revenue) AS total_revenue,
  SUM(total_units_sold) AS total_units,
  ROUND(AVG(avg_order_value), 2) AS avg_order_value
FROM gold_monthly_sales
GROUP BY region
ORDER BY total_revenue DESC;

## New Concept: Window Functions (OVER, PARTITION BY, LAG)

This is the biggest new idea for Power BI users. Window functions let you perform calculations **across related rows** without collapsing the results into groups.

Here’s how to read the syntax:

```sql
LAG(total_revenue) OVER (
  PARTITION BY region, product_category
  ORDER BY report_month
)
```

| Part | What it means | Analogy |
| --- | --- | --- |
| **`LAG(column)`** | “Give me the value from the **previous row**.” | Like PREVIOUSMONTH in DAX |
| **`OVER (...)`** | “Define the **window** of rows I’m looking across.” | The scope of the calculation |
| **`PARTITION BY`** | “Restart the window for each group.” | Like doing a separate calculation per Region |
| **`ORDER BY`** | “Within each partition, arrange rows in this order.” | Determines what “previous” means |

Think of it this way: `PARTITION BY` slices the data into groups (like filter context), and `ORDER BY` defines the sequence within each group.

**`NULLIF(value, 0)`** is a safety net for division: it returns NULL instead of dividing by zero (like `DIVIDE()` in DAX).

---

In [0]:
%sql
-- ============================================================
-- DAX vs SQL: Pattern 5 — Time Intelligence (Month-over-Month)
-- ============================================================
-- DAX: 
--   Revenue MoM% = 
--   DIVIDE(
--     SUM(Sales[total_revenue]) - CALCULATE(SUM(Sales[total_revenue]), PREVIOUSMONTH(Calendar[Date])),
--     CALCULATE(SUM(Sales[total_revenue]), PREVIOUSMONTH(Calendar[Date]))
--   )
--
-- SQL: We use LAG() window function to look at the previous period.
-- LAG() reaches back one row in the specified order.

SELECT
  report_month,
  region,
  total_revenue AS current_revenue,
  LAG(total_revenue) OVER (
    PARTITION BY region, product_category 
    ORDER BY report_month
  ) AS previous_month_revenue,
  ROUND(
    (total_revenue - LAG(total_revenue) OVER (
      PARTITION BY region, product_category 
      ORDER BY report_month
    )) / NULLIF(LAG(total_revenue) OVER (
      PARTITION BY region, product_category 
      ORDER BY report_month
    ), 0) * 100, 1
  ) AS mom_growth_pct
FROM gold_monthly_sales
WHERE product_category = 'Electronics'
ORDER BY region, report_month;

## Window Functions Continued: ROWS BETWEEN (Running Totals)

In Pattern 5, `LAG()` looked at one specific row (the previous one). But window functions can also **accumulate** across multiple rows.

```sql
SUM(total_revenue) OVER (
  PARTITION BY region
  ORDER BY report_month
  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
```

| Part | What it means |
| --- | --- |
| **`SUM(total_revenue) OVER (...)`** | Calculate a running sum across a window of rows |
| **`ROWS BETWEEN UNBOUNDED PRECEDING`** | Start from the very first row in the partition |
| **`AND CURRENT ROW`** | End at the current row |

So for each row, SQL sums everything from the beginning up to that row — a **running total**. This is exactly what `DATESYTD` does in DAX.

---

In [0]:
%sql
-- ============================================================
-- DAX vs SQL: Pattern 6 — Running Total (DATESYTD equivalent)
-- ============================================================
-- DAX:
--   Revenue YTD = CALCULATE(SUM(Sales[total_revenue]), DATESYTD(Calendar[Date]))
--
-- SQL: Use SUM() as a window function with ROWS BETWEEN UNBOUNDED PRECEDING.
-- The window accumulates all prior rows in the partition up to the current row.

SELECT
  report_month,
  region,
  total_revenue,
  SUM(total_revenue) OVER (
    PARTITION BY region 
    ORDER BY report_month
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS revenue_running_total
FROM gold_monthly_sales
WHERE product_category = 'Electronics'
ORDER BY region, report_month;

## Window Functions Continued: Ranking

SQL offers three ranking functions. They all use `OVER (ORDER BY ...)` to define what order to rank in:

| Function | Behaviour | Example (tied at position 2) |
| --- | --- | --- |
| **`RANK()`** | Skips numbers after ties | 1, 2, 2, **4** |
| **`DENSE_RANK()`** | No gaps after ties | 1, 2, 2, **3** |
| **`ROW_NUMBER()`** | Always unique, no ties | 1, 2, 3, 4 |

Use `RANK()` when you want traditional competition-style ranking (like RANKX in DAX). Use `ROW_NUMBER()` when you need exactly one row per rank (useful for deduplication later).

---

In [0]:
%sql
-- ============================================================
-- DAX vs SQL: Pattern 7 — Ranking (RANKX equivalent)
-- ============================================================
-- DAX:
--   Region Rank = RANKX(ALL(Sales[region]), SUM(Sales[total_revenue]))
--
-- SQL: Use RANK() or ROW_NUMBER() window functions.
-- Note: RANK() leaves gaps after ties, DENSE_RANK() does not.

SELECT
  region,
  SUM(total_revenue) AS total_revenue,
  RANK() OVER (ORDER BY SUM(total_revenue) DESC) AS revenue_rank,
  ROW_NUMBER() OVER (ORDER BY SUM(total_revenue) DESC) AS revenue_row_num,
  DENSE_RANK() OVER (ORDER BY SUM(total_revenue) DESC) AS revenue_dense_rank
FROM gold_monthly_sales
GROUP BY region
ORDER BY revenue_rank;

## Window Functions Continued: The Empty OVER () Trick

When you write `OVER ()` with **nothing** inside the parentheses, the window includes **all rows in the result set**. No partitioning, no ordering — just “give me the total across everything.”

This is how we calculate percentage of total:

```sql
SUM(SUM(total_revenue)) OVER () AS grand_total
```

Read it inside-out:
1. `SUM(total_revenue)` — the grouped sum per region/category (from GROUP BY)
2. `SUM(...) OVER ()` — sum of ALL those grouped sums = the grand total

Then we simply divide each group’s revenue by the grand total. This replaces DAX’s `CALCULATE(SUM(...), ALL(Sales))` pattern.

---

In [0]:
%sql
-- ============================================================
-- DAX vs SQL: Pattern 8 — Percentage of Total
-- ============================================================
-- DAX:
--   % of Total Revenue = 
--   DIVIDE(SUM(Sales[total_revenue]), CALCULATE(SUM(Sales[total_revenue]), ALL(Sales)))
--
-- SQL: A window function without PARTITION BY gives you the grand total.
-- SUM(SUM(...)) OVER () means: take each group's SUM, then sum ALL of those.

SELECT
  region,
  product_category,
  SUM(total_revenue) AS category_revenue,
  SUM(SUM(total_revenue)) OVER () AS grand_total,
  ROUND(
    SUM(total_revenue) / SUM(SUM(total_revenue)) OVER () * 100, 
    1
  ) AS pct_of_total
FROM gold_monthly_sales
GROUP BY region, product_category
ORDER BY pct_of_total DESC;

## New Concept: CASE WHEN (Conditional Logic)

`CASE WHEN` is SQL’s if/then/else. It works like DAX’s `SWITCH(TRUE(), ...)` — conditions are evaluated **top to bottom**, and the first match wins.

```sql
CASE
  WHEN condition_1 THEN result_1
  WHEN condition_2 THEN result_2
  ELSE default_result
END
```

You can use it anywhere: in the SELECT list to create calculated columns, in a WHERE clause for complex filtering, or even inside aggregate functions like `SUM(CASE WHEN ... END)`.

---

In [0]:
%sql
-- ============================================================
-- DAX vs SQL: Pattern 9 — Conditional Logic (SWITCH / IF)
-- ============================================================
-- DAX:
--   Performance Band = 
--   SWITCH(TRUE(),
--     [YoY Growth] >= 15, "High Growth",
--     [YoY Growth] >= 5, "Moderate Growth",
--     [YoY Growth] >= 0, "Flat",
--     "Declining"
--   )
--
-- SQL: CASE WHEN expression (evaluated top to bottom, first match wins)

SELECT
  region,
  product_category,
  report_month,
  yoy_growth_pct,
  CASE 
    WHEN yoy_growth_pct >= 15 THEN 'High Growth'
    WHEN yoy_growth_pct >= 5 THEN 'Moderate Growth'
    WHEN yoy_growth_pct >= 0 THEN 'Flat'
    ELSE 'Declining'
  END AS performance_band
FROM gold_monthly_sales
WHERE yoy_growth_pct IS NOT NULL
ORDER BY yoy_growth_pct DESC;

## New Concept: JOINs (Explicit Relationships)

In Power BI, you define a relationship once (e.g., Sales[region] → Customers[region]), and then `RELATED()` traverses it automatically in any measure.

In SQL, you spell out the relationship **every time** using a JOIN:

```sql
FROM gold_monthly_sales s
INNER JOIN dim_customers c
  ON s.region = c.region
```

| Part | What it means |
| --- | --- |
| **`FROM table_a s`** | Start with this table; `s` is a short alias |
| **`INNER JOIN table_b c`** | Bring in rows from another table; `c` is its alias |
| **`ON s.region = c.region`** | The rule for matching rows (the relationship) |

**INNER JOIN** returns only rows that have a match in both tables. Other join types (LEFT, FULL) return unmatched rows too — we’ll cover those later.

The aliases (`s`, `c`) let you write `s.total_revenue` instead of `gold_monthly_sales.total_revenue` — just a time-saver.

---

In [0]:
%sql
-- ============================================================
-- DAX vs SQL: Pattern 10 — RELATED / RELATEDTABLE (JOINs)
-- ============================================================
-- DAX: RELATED() traverses a relationship to pull a column from another table.
--   Customer Segment = RELATED(Customers[segment])
--
-- In Power BI, relationships are defined in the model and traversed
-- automatically. In SQL, you write explicit JOINs.
--
-- SQL: JOIN to bring in data from the dimension table

SELECT 
  s.report_month,
  s.region,
  s.product_category,
  s.total_revenue,
  c.customer_name,
  c.segment
FROM gold_monthly_sales s
INNER JOIN dim_customers c
  ON s.region = c.region
WHERE s.report_month = '2024-12-01'
  AND c.segment = 'Enterprise'
ORDER BY s.total_revenue DESC
LIMIT 10;

---

## Summary: DAX → SQL Cheat Sheet

| DAX Function / Pattern | SQL Equivalent | Key Difference |
| --- | --- | --- |
| `SUM(column)` | `SUM(column)` | Identical syntax |
| `AVERAGE(column)` | `AVG(column)` | Slightly different name |
| `DISTINCTCOUNT(column)` | `COUNT(DISTINCT column)` | SQL uses DISTINCT inside COUNT |
| `CALCULATE(measure, filter)` | `WHERE condition` | No magic context — just a filter clause |
| `ALL(table)` | Window function without PARTITION | Removes group context |
| `RELATED(column)` | `JOIN ... ON` | Explicit relationship traversal |
| `SAMEPERIODLASTYEAR()` | `LAG() OVER (ORDER BY date)` | Window function for time intelligence |
| `DATESYTD()` | `SUM() OVER (... ROWS UNBOUNDED PRECEDING)` | Running total pattern |
| `RANKX()` | `RANK() OVER (ORDER BY ...)` | Window function |
| `DIVIDE(a, b)` | `a / NULLIF(b, 0)` | NULLIF prevents division by zero |
| `IF / SWITCH(TRUE(), ...)` | `CASE WHEN ... END` | Evaluated top to bottom |
| `FORMAT(date, "MMM YYYY")` | `DATE_FORMAT(date, 'MMM yyyy')` | Different format strings |

**The headline:** SQL requires you to be more explicit. DAX’s filter context does a lot of magic behind the scenes. In SQL, you write out your JOINs, your WHERE clauses, and your GROUP BY. The upside is transparency — the logic is right there in front of you and fully debuggable.